In [0]:
%sql
-- Create the target table
create or replace table main_user_target as
select *
from read_files(
    '/Volumes/workspace/default/learning/users_table.csv',
    format => 'csv'
);

-- Display the table
select * from main_user_target;

In [0]:
%sql
-- Create the updated source table
create or replace table update_users_source as
select *
from read_files(
    '/Volumes/workspace/default/learning/update_users_source.csv',
    format => 'csv'
);

-- Display the table
select * from update_users_source;


In [0]:
%sql
-- Merge into
merge into main_user_target as target
using update_users_source as source
on target.id = source.id
when matched and source.status = 'update' then 
  update set
    target.email = source.email,
    target.status = source.status
  when matched and source.status = 'delete' then
    delete
  when not matched then
    insert (id, first_name, email, sign_up_date, status)
    values (source.id, source.first_name, source.email, source.sign_up_date, source.status);

In [0]:
%sql
select * from main_user_target;

In [0]:
%sql
describe history main_user_target;

In [0]:
%sql
-- Time travel (if needed)
select * from main_user_target version as of 1;

#Let´s see how schema enforcement works

In [0]:
%sql
-- Create the new users source table
create or replace table new_users_source as
select *
from read_files(
    '/Volumes/workspace/default/learning/new_users_source.csv',
    format => 'csv'
);

-- Display the table
select * from new_users_source;

In [0]:
%sql
-- if we apply the merge into statement, it´ll fail because the schema does not match, namely, target does not have a country col
-- Then we have to define a new merge into statement with schema evolution
merge with schema evolution into main_user_target as target
using new_users_source as source
on target.id = source.id
when matched and source.status = 'update' then 
  update set
    target.email = source.email,
    target.status = source.status
  when matched and source.status = 'delete' then
    delete
  when not matched and source.status = 'new' then
    insert (id, first_name, email, sign_up_date, status, country)
    values (source.id, source.first_name, source.email, source.sign_up_date, source.status, source.country);

In [0]:
%sql
select * from main_user_target order by id;